In [0]:
from pyspark.sql import functions as F

CATALOG = "voebem"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

TABELA_BRONZE_VRA = f"{CATALOG}.{SCHEMA_BRONZE}.vra"
TABELA_SILVER_VRA = f"{CATALOG}.{SCHEMA_SILVER}.vra"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}"
)

In [0]:
df_bronze = spark.table(TABELA_BRONZE_VRA)

print(f"Registros Bronze: {df_bronze.count():,}")
print(f"Quantidade de colunas: {len(df_bronze.columns)}")

df_bronze.printSchema()

display(df_bronze)

In [0]:
df_silver = df_bronze

print(f"Registros Silver: {df_silver.count():,}")
print(f"Quantidade de colunas: {len(df_silver.columns)}")

%md
### Transformação de dados - Tratamento de datas

Convertemos as colunas de data/hora do formato string para o tipo `TIMESTAMP`, utilizando a função `try_cast` para garantir resiliência contra valores nulos ou inválidos:

* **`partida_prevista_ts`**: Data/hora prevista de partida.
* **`partida_real_ts`**: Data/hora real de partida.
* **`chegada_prevista_ts`**: Data/hora prevista de chegada.
* **`chegada_real_ts`**: Data/hora real de chegada.

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "partida_prevista_ts",
        F.expr("try_cast(partida_prevista AS TIMESTAMP)")
    )
    .withColumn(
        "partida_real_ts",
        F.expr("try_cast(partida_real AS TIMESTAMP)")
    )
    .withColumn(
        "chegada_prevista_ts",
        F.expr("try_cast(chegada_prevista AS TIMESTAMP)")
    )
    .withColumn(
        "chegada_real_ts",
        F.expr("try_cast(chegada_real AS TIMESTAMP)")
    )
)

### Transformação de dados - Cálculo de atraso na partida e na chegada

Calculamos a diferença em minutos entre o horário real e o horário previsto para as etapas de partida e chegada:

* **`atraso_partida_min`**: Diferença em minutos entre a partida real e a prevista (`(unix_timestamp(partida_real_ts) - unix_timestamp(partida_prevista_ts)) / 60`).
* **`atraso_chegada_min`**: Diferença em minutos entre a chegada real e a prevista (`(unix_timestamp(chegada_real_ts) - unix_timestamp(chegada_prevista_ts)) / 60`).

 **Observação:** Cada cálculo só é executado se ambos os campos de timestamp correspondentes forem preenchidos (não nulos).

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "atraso_partida_min",
        F.when(
            F.col("partida_prevista_ts").isNotNull()
            & F.col("partida_real_ts").isNotNull(),
            (
                F.unix_timestamp("partida_real_ts") # Valor em segundos
                - F.unix_timestamp("partida_prevista_ts")
            ) / 60 # Transforma em minutos
        )
    )
    .withColumn(
        "atraso_chegada_min",
        F.when(
            F.col("chegada_prevista_ts").isNotNull()
            & F.col("chegada_real_ts").isNotNull(),
            (
                F.unix_timestamp("chegada_real_ts")
                - F.unix_timestamp("chegada_prevista_ts")
            ) / 60
        )
    )
)

In [0]:
print(f"Registros Silver: {df_silver.count():,}") # Não alteramos
print(f"Colunas Silver: {len(df_silver.columns)}")

display(
    df_silver.select(
        "icao_empresa",
        "numero_voo",
        "icao_origem",
        "icao_destino",
        "partida_prevista_ts",
        "partida_real_ts",
        "atraso_partida_min",
        "chegada_prevista_ts",
        "chegada_real_ts",
        "atraso_chegada_min",
        "situacao_voo"
    )
)

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_VRA)
)